In [1]:
import ast
import os

In [2]:
from urllib.parse import urlparse

def get_filename(url: str):
    if url[-4:] == ".git":
        url = url[:-4]
        
    parts = urlparse(url).path.strip("/").split("/")

    if len(parts) >= 2:
        username = parts[0]
        project_name = parts[1]
        result = f"{username}-{project_name}"
        return result

In [3]:
repo = "https://github.com/princ3kr/Notebook-LM-Mini"

filename = get_filename(repo)
dir = f"../temp/{filename}"

if os.path.isdir(dir):
    print("Directory already exists")

Directory already exists


In [4]:
import subprocess

if os.path.isdir(dir):
    print("Directory already exists")
else:
    try:
        subprocess.run(["git", "clone", repo, dir], check=True)
        print("Clone successful!")
    except subprocess.CalledProcessError as e:
        print(f"Git command failed with error: {e}")


Directory already exists


In [5]:
def _expr_name(node):
    if isinstance(node, ast.Name):
        return node.id
    if isinstance(node, ast.Attribute):
        parent = _expr_name(node.value)
        return f"{parent}.{node.attr}" if parent else node.attr
    if isinstance(node, ast.Call):
        return _expr_name(node.func)
    if isinstance(node, ast.Subscript):
        return _expr_name(node.value)
    return None

def _collect_calls(node, caller):
    calls = []
    nested_def_types = (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)

    for child in ast.walk(node):
        # Skip the function node itself
        if child is node:
            continue
        # Don't descend into nested function/class definitions
        if isinstance(child, nested_def_types):
            continue
        if isinstance(child, ast.Call):
            callee = _expr_name(child.func)
            if callee:
                calls.append({
                    "caller": caller,
                    "callee": callee,
                    "line": getattr(child, "lineno", None)
                })
    return calls

def parse_file(source_code, filepath=None):
    tree = ast.parse(source_code)
    import_modules, import_names, classes, functions = [], [], [], []
    imports, methods, calls, inheritance = [], [], [], []

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                import_modules.append(alias.name)
                import_names.append([alias.asname or alias.name.split(".")[0]])
                imports.append({"module": alias.name, "name": alias.name.split(".")[0], "alias": alias.asname, "line": node.lineno})

        if isinstance(node, ast.ImportFrom):
            import_modules.append(node.module)
            import_names.append([alias.asname or alias.name for alias in node.names])
            for alias in node.names:
                imports.append({"module": node.module or "", "name": alias.name, "alias": alias.asname, "line": node.lineno})

        if isinstance(node, ast.ClassDef):
            class_qname = node.name if filepath is None else f"{filepath}::{node.name}"
            bases = [base for base in (_expr_name(base) for base in node.bases) if base]
            classes.append({"name": node.name, "qualified_name": class_qname, "bases": bases, "line_start": node.lineno, "line_end": node.end_lineno})
            for base in bases:
                inheritance.append({"class": node.name, "qualified_name": class_qname, "base": base, "line": node.lineno})

            for child in node.body:
                if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    method_qname = f"{class_qname}.{child.name}"
                    method = {"name": child.name, "qualified_name": method_qname, "class_name": node.name, "line_start": child.lineno, "line_end": child.end_lineno}
                    methods.append(method)
                    functions.append(method)
                    calls.extend(_collect_calls(child, method_qname))

        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            parent_class = next((cls.name for cls in ast.walk(tree) if isinstance(cls, ast.ClassDef) and node in cls.body), None)
            if parent_class:
                continue
            func_qname = node.name if filepath is None else f"{filepath}::{node.name}"
            function = {"name": node.name, "qualified_name": func_qname, "class_name": None, "line_start": node.lineno, "line_end": node.end_lineno}
            functions.append(function)
            calls.extend(_collect_calls(node, func_qname))

    return {"import_modules": import_modules, "import_names": import_names, "imports": imports, "classes": classes, "functions": functions, "methods": methods, "calls": calls, "inheritance": inheritance}

In [6]:
ignores = { ".git", ".gitignore", ".lock", ".venv", "__pycache__", "node_modules", ".vscode", "pyproject.toml", "__init__.py", ".python-version", "requirements.txt" }

def get_structure(directory: str):
    for root, dirs, files in os.walk(directory):
        dirs[:] = sorted([d for d in dirs if d not in ignores])

        rel_dir = os.path.relpath(root, directory)
        
        if rel_dir == ".":
            level = 0
            display_name = os.path.basename(os.path.abspath(directory))
        else:
            level = rel_dir.count(os.sep) + 1
            display_name = os.path.basename(root)

        indent = "│   " * (level - 1) if level > 0 else ""
        connector = "├── " if level > 0 else ""
        print(f"{indent}{connector}{display_name}/")

        file_indent = "│   " * level
        sorted_files = sorted(files)
        for i, name in enumerate(sorted_files):
            if name not in ignores or name == "README.md":
                file_connector = "└── " if (i == len(sorted_files) - 1 and not dirs) else "├── "
                print(f"{file_indent}{file_connector}{name}")

def get_files(directory: str):
    inventory = {}
    for root, dirs, files in os.walk(directory):
        rel_dir = os.path.relpath(root, directory)
        if rel_dir == ".":
            rel_dir = ""
        
        for name in files:
            if (name not in ignores) and (name == "README.md" or name.endswith(".py")):
                full_path = os.path.join(root, name)

                with open(full_path, "r", encoding="utf-8") as f:
                    content = f.read()
                    posix_path = os.path.join(rel_dir, name).replace("\\", "/")

                    parsed_structure = {"import_modules": [], "import_names": [], "imports": [], "classes": [], "functions": [], "methods": [], "calls": [], "inheritance": []}
                    if name.endswith(".py"):
                        try:
                            parsed_structure = parse_file(content, filepath=posix_path)
                        except SyntaxError:
                            pass

                    parsed_structure['content'] = content
                    
                    inventory[posix_path] = parsed_structure

    return inventory


In [7]:
# get_structure(dir)
files = get_files(directory=dir)

In [8]:
for imports in files['src/database/neo4j_conn.py']:
    print(imports)

import_modules
import_names
imports
classes
functions
methods
calls
inheritance
content


In [ ]:
import networkx as nx
from pyvis.network import Network
from neo4j import GraphDatabase
import json

class BuildGraph:
    def __init__(self, files: dict):
        self.G = nx.DiGraph()
        self.files = files
        self.name_index = self.build_name_index()
        self.module_index = self.build_module_index()

    def build_name_index(self) -> dict:
        name_index = {}
        for filepath in self.files:
            for classname in self.files.get(filepath, {}).get("classes", []):
                name_index[classname['name']] = filepath
            for funcname in self.files.get(filepath, {}).get("functions", []):
                name_index[funcname['name']] = filepath
        return name_index

    def build_module_index(self) -> dict:
        """Creates an index mapping python module paths (e.g. 'utils.helpers') to file paths"""
        module_index = {}
        for filepath in self.files:
            if filepath.endswith(".py"):
                # Convert 'src/utils/helpers.py' -> 'src.utils.helpers' and 'utils.helpers'
                parts = filepath[:-3].split("/")
                for i in range(len(parts)):
                    module_name = ".".join(parts[i:])
                    module_index[module_name] = filepath
        return module_index

    def get_path(self) -> dict:
        cleared_files = {}
        for filepath in self.files:
            imports = []
            modules = self.files[filepath].get("import_modules", [])
            names = self.files[filepath].get("import_names", [])

            for imp, imp_names in zip(modules, names):
                if not imp:
                    continue

                if imp in self.module_index:
                    imports.append(self.module_index[imp])
                else:
                    for name in imp_names:
                        if name in self.name_index:
                            imports.append(self.name_index[name])
                            
            cleared_files[filepath] = list(set(imports))
            
        return cleared_files

    def get_node_style(self, filepath):
        """Dynamically assigns colors based on top-level directory"""
        parts = filepath.split("/")
        top_level = parts[0] if len(parts) > 1 else "root"

        colors = [
            {"background": "#3b82f6", "border": "#60a5fa", "highlight": "#93c5fd"},
            {"background": "#10b981", "border": "#34d399", "highlight": "#6ee7b7"},
            {"background": "#f59e0b", "border": "#fbbf24", "highlight": "#fcd34d"},
            {"background": "#8b5cf6", "border": "#a78bfa", "highlight": "#c4b5fd"},
            {"background": "#ec4899", "border": "#f472b6", "highlight": "#fbcfe8"},
            {"background": "#ef4444", "border": "#f87171", "highlight": "#fca5a5"},
        ]
        
        color_idx = hash(top_level) % len(colors)
        return colors[color_idx]

    def build(self):
        cleared_path = self.get_path()
        for source, targets in cleared_path.items():
            style = self.get_node_style(source)
            self.G.add_node(
                source, 
                label=source.split("/")[-1],
                title=source,
                color=style,
                shadow=True,
                shape="dot",
                size=25,
                font={"color": "white", "size": 14, "face": "Segoe UI, Tahoma, Geneva, Verdana, sans-serif"}
            )
            for target in targets:
                self.G.add_edge(source, target)

    def _code_indexes(self):
        class_index, function_index, method_index, caller_class = {}, {}, {}, {}
        for path, data in self.files.items():
            for cls in data.get("classes", []):
                class_index.setdefault(cls["name"], cls["qualified_name"])
            for fn in data.get("functions", []):
                function_index.setdefault(fn["name"], fn["qualified_name"])
                function_index[fn["qualified_name"]] = fn["qualified_name"]
                if fn.get("class_name"):
                    method_index[(fn["class_name"], fn["name"])] = fn["qualified_name"]
                    caller_class[fn["qualified_name"]] = fn["class_name"]
        return class_index, function_index, method_index, caller_class

    def _resolve_call(self, caller, callee, class_index, function_index, method_index, caller_class):
        # Return BOTH type and full qualified name
        
        # Check direct qualified name match
        if callee in function_index:
            return "function", function_index[callee]  # Already qualified
        if callee in class_index:
            return "class", class_index[callee]  # Already qualified
        
        # Check self.method() calls
        current_class = caller_class.get(caller)
        if current_class and callee.startswith("self."):
            method_name = callee.split(".")[-1]
            target = method_index.get((current_class, method_name))
            if target:
                return "function", target  # Qualified from method_index
        
        # Check short name (just the function name, not qualified)
        short_name = callee.split(".")[-1]
        if short_name in function_index:
            # Return the QUALIFIED name, not short name
            return "function", function_index[short_name]
        if short_name in class_index:
            return "class", class_index[short_name]
        
        # External call
        return "external", callee

    def push_to_neo4j(self, uri, user, password):
        driver = GraphDatabase.driver(uri, auth=(user, password))
        repo_id = filename
        class_index, function_index, method_index, caller_class = self._code_indexes()

        with driver.session() as session:
            for path, data in self.files.items():
                imports = [imp.get("name") for imp in data.get("imports", []) if imp.get("name")]
                session.run('''
                    MERGE (r:Repo {repo_id: $repo_id})
                    MERGE (f:File {path: $path, repo_id: $repo_id})
                    MERGE (r)-[:CONTAINS]->(f)
                    SET f.name = $name,
                        f.classes = $classes,
                        f.imports = $imports,
                        f.functions = $functions
                ''', repo_id=repo_id, path=path, name=path.split('/')[-1],
                     classes=[c['name'] for c in data.get("classes", [])],
                     imports=imports,
                     functions=[f['name'] for f in data.get("functions", [])])

                for cls in data.get("classes", []):
                    session.run('''
                        MATCH (f:File {path: $path, repo_id: $repo_id})
                        MERGE (c:Class {qualified_name: $qualified_name, repo_id: $repo_id})
                        SET c.name = $name, c.path = $path, c.bases = $bases,
                            c.line_start = $line_start, c.line_end = $line_end
                        MERGE (f)-[:DEFINES_CLASS]->(c)
                    ''', repo_id=repo_id, path=path, **cls)

                for fn in data.get("functions", []):
                    session.run('''
                        MATCH (f:File {path: $path, repo_id: $repo_id})
                        MERGE (fn:Function {qualified_name: $qualified_name, repo_id: $repo_id})
                        SET fn.name = $name, fn.path = $path, fn.class_name = $class_name,
                            fn.line_start = $line_start, fn.line_end = $line_end
                        WITH f, fn
                        OPTIONAL MATCH (c:Class {name: $class_name, path: $path, repo_id: $repo_id})
                        FOREACH (_ IN CASE WHEN c IS NULL THEN [1] ELSE [] END | MERGE (f)-[:DEFINES_FUNCTION]->(fn))
                        FOREACH (_ IN CASE WHEN c IS NULL THEN [] ELSE [1] END | MERGE (c)-[:HAS_METHOD]->(fn))
                    ''', repo_id=repo_id, path=path, **fn)

                for imp in data.get("imports", []):
                    session.run('''
                        MATCH (f:File {path: $path, repo_id: $repo_id})
                        MERGE (i:Import {path: $path, name: $name, module: $module, repo_id: $repo_id})
                        SET i.alias = $alias, i.line = $line
                        MERGE (f)-[:IMPORTS_SYMBOL]->(i)
                    ''', repo_id=repo_id, path=path, **imp)

            for source, target in self.G.edges():
                session.run('''
                    MATCH (a:File {path: $source, repo_id: $repo_id})
                    MATCH (b:File {path: $target, repo_id: $repo_id})
                    MERGE (a)-[:IMPORTS]->(b)
                ''', source=source, target=target, repo_id=repo_id)

            for path, data in self.files.items():
                for rel in data.get("inheritance", []):
                    base_type, target = ("class", class_index[rel["base"]]) if rel["base"] in class_index else ("external", rel["base"])
                    if base_type == "class":
                        session.run('''
                            MATCH (c:Class {qualified_name: $class_qname, repo_id: $repo_id})
                            MATCH (base:Class {qualified_name: $base_qname, repo_id: $repo_id})
                            MERGE (c)-[:INHERITS_FROM {line: $line}]->(base)
                        ''', repo_id=repo_id, class_qname=rel["qualified_name"], base_qname=target, line=rel["line"])
                    else:
                        session.run('''
                            MATCH (c:Class {qualified_name: $class_qname, repo_id: $repo_id})
                            MERGE (ext:ExternalSymbol {name: $base_name, repo_id: $repo_id})
                            MERGE (c)-[:INHERITS_EXTERNAL {line: $line}]->(ext)
                        ''', repo_id=repo_id, class_qname=rel["qualified_name"], base_name=target, line=rel["line"])

                for call in data.get("calls", []):
                    target_type, target = self._resolve_call(call["caller"], call["callee"], class_index, function_index, method_index, caller_class)
                    if target_type == "function":
                        session.run('''
                            MATCH (caller:Function {qualified_name: $caller, repo_id: $repo_id})
                            MATCH (callee:Function {qualified_name: $callee, repo_id: $repo_id})
                            MERGE (caller)-[:CALLS {line: $line}]->(callee)
                        ''', repo_id=repo_id, caller=call["caller"], callee=target, line=call["line"])
                    elif target_type == "class":
                        session.run('''
                            MATCH (caller:Function {qualified_name: $caller, repo_id: $repo_id})
                            MATCH (callee:Class {qualified_name: $callee, repo_id: $repo_id})
                            MERGE (caller)-[:INSTANTIATES {line: $line}]->(callee)
                        ''', repo_id=repo_id, caller=call["caller"], callee=target, line=call["line"])
                    else:
                        session.run('''
                            MATCH (caller:Function {qualified_name: $caller, repo_id: $repo_id})
                            MERGE (ext:ExternalSymbol {name: $callee, repo_id: $repo_id})
                            MERGE (caller)-[:CALLS_EXTERNAL {line: $line}]->(ext)
                        ''', repo_id=repo_id, caller=call["caller"], callee=target, line=call["line"])
                
    def show(self, filename="graph.html"):
        net = Network(
            directed=True, notebook=True, cdn_resources='remote', 
            height="750px", width="100%", bgcolor="#111827", font_color="white"
        )
        net.from_nx(self.G)
        options = {
            "edges": {"color": {"inherit": "from", "opacity": 0.5}, "smooth": {"type": "continuous", "roundness": 0.5}, "arrows": {"to": {"enabled": True, "scaleFactor": 0.5}}, "width": 1.5},
            "physics": {"forceAtlas2Based": {"gravitationalConstant": -60, "centralGravity": 0.005, "springLength": 150, "springStrength": 0.08, "damping": 0.4, "avoidOverlap": 0.5}, "maxVelocity": 50, "minVelocity": 0.1, "solver": "forceAtlas2Based", "stabilization": {"enabled": True, "iterations": 1000}},
            "interaction": {"hover": True, "navigationButtons": True, "multiselect": True, "tooltipDelay": 200}
        }
        net.set_options(json.dumps(options))
        return net.write_html(filename)


In [10]:
builder = BuildGraph(files)
builder.build()
# builder.show()

In [11]:
# builder.push_to_neo4j("bolt://localhost:7687", "neo4j", "pk142145")

In [12]:
import os
import uuid
from dotenv import load_dotenv
from qdrant_client import QdrantClient

load_dotenv()
os.environ['HF_HOME'] = '/p/repo-2-graph/models'

class VectorStore:
    def __init__(self, files: dict, collection_name: str = 'prototype'):
        self.files = files
        self.collection_name = collection_name
        self.chunks = []
        
        url = os.getenv("QDRANT_END_POINT")
        api_key = os.getenv("QDRANT_API_KEY")
        
        if not url or not api_key:
            raise ValueError("Please set QDRANT_END_POINT and QDRANT_API_KEY in your .env file!")
            
        self.client = QdrantClient(url=url, api_key=api_key, timeout=60)
        
        self.client.set_model("jinaai/jina-embeddings-v2-base-code")
        self.client.set_sparse_model("Qdrant/bm25")
        
        if not self.client.collection_exists(self.collection_name):
            self.client.create_collection(
                collection_name=self.collection_name,
                vectors_config=self.client.get_fastembed_vector_params(),
                sparse_vectors_config=self.client.get_fastembed_sparse_vector_params()
            )
            
        # Create a payload index on the "path" field so we can filter by it
        self.client.create_payload_index(
            collection_name=self.collection_name,
            field_name="path",
            field_schema="keyword"
        )


    def build(self, max_module_lines=100, overlap=5):
        self.chunks = []
        
        for file in self.files.keys():
            content = self.files[file]['content']
            lines = content.split('\n')
            classes = self.files[file]['classes']
            functions = self.files[file]['functions']
            
            # ========== CHUNK 1: Classes ==========
            for cls in classes:
                start_line = cls['line_start'] - 1  # Convert to 0-indexed
                end_line = cls['line_end']  # Already exclusive in AST
                
                chunk = "\n".join(lines[start_line:end_line])
                metadata = {
                    "path": file.replace("\\", "/"),
                    "filename": file.split("/")[-1],
                    "function_name": cls['name'],
                    "class_name": cls['name'],
                    "chunk_type": "class",
                    "line_start": start_line + 1,  # Convert back to 1-indexed for display
                    "line_end": end_line,
                    "methods": [f['name'] for f in functions 
                            if f.get('class_name') == cls['name']]
                }
                self.chunks.append([chunk, metadata])
            
            # ========== CHUNK 2: Functions ==========
            for func in functions:
                # Skip if it's a method (already in class chunk)
                if func.get('class_name'):
                    continue
                
                # Find decorator context
                start_line = func['line_start'] - 1
                while start_line > 0 and lines[start_line - 1].strip().startswith('@'):
                    start_line -= 1
                
                end_line = func['line_end']
                
                chunk = "\n".join(lines[start_line:end_line])
                metadata = {
                    "path": file.replace("\\", "/"),
                    "filename": file.split("/")[-1],
                    "function_name": func['name'],
                    "class_name": "module_level",
                    "chunk_type": "function",
                    "line_start": start_line + 1,
                    "line_end": end_line
                }
                self.chunks.append([chunk, metadata])
            
            # ========== CHUNK 3: Module-level (imports, constants) ==========
            # Collect all lines that are NOT in functions or classes
            function_ranges = [(f['line_start'] - 1, f['line_end']) for f in functions]
            class_ranges = [(c['line_start'] - 1, c['line_end']) for c in classes]
            all_ranges = function_ranges + class_ranges
            
            # Collect SEPARATE module segments (imports, constants, etc.)
            module_lines = []
            for i, line in enumerate(lines):
                in_func_or_class = any(start <= i < end for start, end in all_ranges)
                if not in_func_or_class and line.strip():  # Skip empty lines
                    module_lines.append(i)

            if module_lines:
                # Chunk module code in 50-line segments to avoid huge blobs
                for chunk_start_idx in range(0, len(module_lines), 50):
                    chunk_indices = module_lines[chunk_start_idx:chunk_start_idx+50]
                    start_line = chunk_indices[0]
                    end_line = chunk_indices[-1] + 1
                    
                    chunk = "\n".join(lines[start_line:end_line])
                    metadata = {
                        "path": file.replace("\\", "/"),
                        "filename": file.split("/")[-1],
                        "function_name": f"module_segment_{chunk_start_idx//50}",
                        "class_name": "module_level",
                        "chunk_type": "module",
                        "line_start": start_line + 1,
                        "line_end": end_line
                    }
                    self.chunks.append([chunk, metadata])

    def push(self, batch_size: int = 32):
        if not self.chunks:
            print("No chunks to push. Run build() first.")
            return

        documents = [c[0] for c in self.chunks]
        metadatas = [c[1] for c in self.chunks]
        ids = [
            str(uuid.uuid5(uuid.NAMESPACE_DNS, f"{m['path']}:{m['function_name']}:{i}"))
            for i, m in enumerate(metadatas)
        ]

        total = len(documents)
        for start in range(0, total, batch_size):
            end = min(start + batch_size, total)
            print(f"Pushing batch {start}-{end} / {total}...")
            
            self.client.add(
                collection_name=self.collection_name,
                documents=documents[start:end],
                metadata=metadatas[start:end],
                ids=ids[start:end]
            )
        
        print(f"Successfully pushed {total} chunks to Qdrant!")

    def search(self, query: str, query_filter=None, top_k: int = 5):
        results = self.client.query(
            collection_name=self.collection_name,
            query_text=query,
            query_filter=query_filter,
            limit=top_k
        )

        return {
            "documents": [[hit.document for hit in results]],
            "metadatas": [[hit.metadata for hit in results]],
            "ids": [[str(hit.id) for hit in results]]
        }


p:\repo-2-graph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
client = VectorStore(files, collection_name='R2Mapper')
# client.build()

In [15]:
# client.push()

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Literal, Optional
from sentence_transformers import SentenceTransformer
from fuzzywuzzy import process, fuzz
from qdrant_client.models import Filter, FieldCondition, MatchAny
from neo4j import GraphDatabase
import re
import numpy as np
import time

VALID_FILE_PROPERTIES = {
    'name', 'path', 'functions', 'classes', 'imports', 'repo_id', 
    'qualified_name', 'class_name', 'bases', 'module', 'alias', 'line', 
    'line_start', 'line_end'
}

class CypherQuery(BaseModel):
    cypher: str = Field(..., description="Cypher query for retrieving relationships within the nodes")

class RouterDecision(BaseModel):
    decision: Literal['hybrid', 'graph_only'] = Field(
        ..., 
        description="graph_only ONLY for pure structural/dependency/topology questions. hybrid for everything else."
    )

class QueryEngine:
    def __init__(self, repo_id: str, db_client, uri: str, user: str, password: str, llm: ChatGroq):
        self.db_client = db_client
        self.repo_id = repo_id
        self.llm = llm
        self.graph_driver = GraphDatabase.driver(uri, auth=(user, password))
        self._embed_model = None
        self._file_index = None
        self._router_cache: dict[str, RouterDecision] = {}
        self._graph_cache: dict[str, dict | None] = {}

        self.query_templates = self._init_templates()

        self.min_result_count = 1  # Minimum files expected for scoped search
        self.max_result_variance = 0.5  # Flag if results seem incomplete
    
    def _init_templates(self) -> dict:
        """Initialize Cypher query templates for common patterns"""
        return {
            "multi_hop_imports": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})
                MATCH (source:File {{name: '{source}'}})-[:IMPORTS]->(via:File {{name: '{via}'}})
                MATCH (via:File)-[:IMPORTS]->(target:File)
                RETURN DISTINCT target.path as indirect_dependency
                ORDER BY target.path
            """,
            
            "leaf_nodes": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})-[:CONTAINS]->(f:File)
                WHERE NOT (f)-[:IMPORTS]->()
                RETURN f.path as leaf_file
                ORDER BY f.path
            """,
            
            "most_dependencies": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})-[:CONTAINS]->(f:File)
                MATCH (f)-[:IMPORTS]->(deps:File)
                WITH f, count(DISTINCT deps) as dep_count
                ORDER BY dep_count DESC
                LIMIT 5
                RETURN f.path as file, dep_count as dependency_count
            """,
            
            "direct_imports": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})
                MATCH (f:File {{name: '{filename}'}})-[:IMPORTS]->(imported:File)
                RETURN imported.path as imported_file
                ORDER BY imported.path
            """,
            
            "reverse_lookup": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})
                MATCH (importer:File)-[:IMPORTS]->(target:File {{name: '{filename}'}})
                RETURN importer.path as file_that_imports
                ORDER BY importer.path
            """,
            
            "transitive_dependencies": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})
                MATCH path=(source:File {{name: '{source}'}})-[:IMPORTS*1..3]->(target:File)
                WHERE source <> target
                RETURN DISTINCT target.path as transitive_dep
                ORDER BY target.path
            """,
            
            "file_structure": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})-[:CONTAINS]->(f:File {{name: '{filename}'}})
                MATCH (f)-[:DEFINES_CLASS]->(c:Class)
                MATCH (f)-[:DEFINES_FUNCTION]->(fn:Function)
                RETURN f.path as file_path, 
                       collect(DISTINCT c.name) as classes, 
                       collect(DISTINCT fn.name) as functions
            """,
            
            "call_graph": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})
                MATCH (caller:Function {{name: '{function_name}'}})-[:CALLS]->(callee:Function)
                RETURN callee.qualified_name as called_function
                ORDER BY called_function
            """,
            
            "class_hierarchy": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})
                MATCH (c:Class {{name: '{class_name}'}})-[:INHERITS_FROM]->(parent:Class)
                RETURN parent.qualified_name as parent_class
                ORDER BY parent_class
            """,
            
            "all_files": """
                MATCH (r:Repo {{repo_id: '{repo_id}'}})-[:CONTAINS]->(f:File)
                RETURN f.path as file_path
                ORDER BY f.path
            """
        }

    def _match_template(self, query: str) -> Optional[tuple[str, dict]]:
        """
        Pattern match query against known templates.
        Returns: (template_name, extracted_params) or None
        """
        # Multi-hop: "Which files does X indirectly depend on through Y?"
        match = re.search(
            r'(?:which\s+)?files?\s+(?:does?\s+)?(\w+\.py)\s+(?:indirectly\s+)?depend(?:s|ency)?\s+on\s+through\s+(\w+\.py)',
            query.lower()
        )
        if match:
            return ("multi_hop_imports", {
                "repo_id": self.repo_id,
                "source": match.group(1),
                "via": match.group(2)
            })
        
        # Leaf nodes: "Which files have no imports?" / "Files with no imports"
        if re.search(r'leaf\s+node|no\s+import|do\s+not\s+import\s+any', query.lower()):
            return ("leaf_nodes", {"repo_id": self.repo_id})
        
        # Most dependencies: "Which service has the most dependencies?"
        if re.search(r'most\s+dependen|highest\s+dependen|most\s+import', query.lower()):
            return ("most_dependencies", {"repo_id": self.repo_id})
        
        # Direct imports: "What does X import?" / "X imports what?"
        match = re.search(r'(?:what\s+(?:does|do)\s+)?(\w+\.py)\s+import', query.lower())
        if match:
            return ("direct_imports", {
                "repo_id": self.repo_id,
                "filename": match.group(1)
            })
        
        # Reverse lookup: "Which files import X?"
        match = re.search(r'(?:which|what)\s+files?\s+import\s+(\w+\.py)', query.lower())
        if match:
            return ("reverse_lookup", {
                "repo_id": self.repo_id,
                "filename": match.group(1)
            })
        
        # Transitive: "What are the transitive dependencies of X?"
        match = re.search(r'transitive\s+dependenc(?:y|ies).*?(\w+\.py)', query.lower())
        if match:
            return ("transitive_dependencies", {
                "repo_id": self.repo_id,
                "source": match.group(1)
            })
        
        # File structure: "What is in X?" / "Structure of X"
        match = re.search(r'(?:structure|content|what\s+is\s+in)\s+(\w+\.py)', query.lower())
        if match:
            return ("file_structure", {
                "repo_id": self.repo_id,
                "filename": match.group(1)
            })
        
        # Call graph: "What does function X call?"
        match = re.search(r'(?:what\s+(?:does|do)\s+)?(?:function\s+)?(\w+)\s+call', query.lower())
        if match:
            return ("call_graph", {
                "repo_id": self.repo_id,
                "function_name": match.group(1)
            })
        
        # Class hierarchy: "What does class X inherit?"
        match = re.search(r'(?:what\s+(?:does|do)\s+)?(?:class\s+)?(\w+)\s+inherit', query.lower())
        if match:
            return ("class_hierarchy", {
                "repo_id": self.repo_id,
                "class_name": match.group(1)
            })
        
        return None

    def _execute_template(self, template_name: str, params: dict) -> dict | None:
        """Execute a template query safely"""
        try:
            cypher = self.query_templates[template_name].format(**params)
            
            with self.graph_driver.session() as session:
                result = session.run(cypher)
                data = [record.data() for record in result]
            
            return {
                "is_fallback": False,
                "data": data,
                "response": cypher,
                "method": "template",
                "template_name": template_name,
                "timestamp": time.time()
            }
        except Exception as e:
            print(f"[Template Error] {template_name}: {str(e)}")
            return None


    def graph_search(self, query: str) -> dict | None:
        """
        Enhanced graph search with better Cypher generation prompt
        """
        if query in self._graph_cache:
            return self._graph_cache[query]
        
        # Try template first
        template_match = self._match_template(query)
        if template_match:
            template_name, params = template_match
            result = self._execute_template(template_name, params)
            if result:
                self._graph_cache[query] = result
                return result
        
        structured_llm = self.llm.with_structured_output(CypherQuery)
        
        index = self._load_file_index()
        sample = index[:15]
        file_list = "\n".join(f"- {f['name']} ({f['path']})" for f in sample)
        
        system_content = f"""You are a Cypher expert for Neo4j. Generate ONLY valid, working queries.
            CRITICAL RULES:
            1. ALWAYS start with: MATCH (r:Repo {{repo_id: '{self.repo_id}'}})
            2. NEVER use properties not in VALID_PROPERTIES list
            3. For multi-hop queries (A -> B -> C), use proper MATCH chains with intermediate variables
            4. ALWAYS use DISTINCT if returning multiple instances
            5. ALWAYS ORDER BY results for consistency
            6. For "files that depend on X", use reverse relationship direction
            7. For "transitive" queries, use variable-length patterns: -[:IMPORTS*1..3]->
            8. Return DISTINCT paths to avoid duplicates
            9. Always filter by repo_id to scope queries
            10. If unsure about relationship direction, return sentinel query

            SCHEMA:
            Nodes:
            - Repo(repo_id)
            - File(name, path, functions, classes, imports, repo_id)
            - Class(name, qualified_name, path, bases, line_start, line_end, repo_id)
            - Function(name, qualified_name, path, class_name, line_start, line_end, repo_id)
            - Import(name, module, alias, path, line, repo_id)
            - ExternalSymbol(name, repo_id)

            Edges (all have repo_id):
            - (Repo)-[:CONTAINS]->(File)
            - (File)-[:IMPORTS]->(File)
            - (File)-[:IMPORTS_SYMBOL]->(Import)
            - (File)-[:DEFINES_CLASS]->(Class)
            - (File)-[:DEFINES_FUNCTION]->(Function)
            - (Class)-[:HAS_METHOD]->(Function)
            - (Class)-[:INHERITS_FROM]->(Class)
            - (Class)-[:INHERITS_EXTERNAL]->(ExternalSymbol)
            - (Function)-[:CALLS]->(Function)
            - (Function)-[:INSTANTIATES]->(Class)
            - (Function)-[:CALLS_EXTERNAL]->(ExternalSymbol)

            VALID PROPERTIES: {', '.join(sorted(VALID_FILE_PROPERTIES))}

            SAMPLE FILES IN REPO:
            {file_list}

            EXAMPLES OF CORRECT QUERIES:
            Example 1: Multi-hop (A -> B -> C)
            Query: "What does X import transitively?"
            Example 2: Leaf nodes (no outgoing edges)
            Query: "Which files have no imports?"
            Example 3: Reverse lookup (who imports X)
            Query: "Which files import X?"
            Example 4: Variable-length (transitive)
            Query: "Transitive dependencies of X?"

            IMPORTANT:
            - For behavioral questions (what does code do), return sentinel immediately
            - Never hardcode file names except when explicitly mentioned in question
            - Use DISTINCT to avoid duplicates
            - Filter by repo_id in first MATCH
            - Return file paths, not File nodes themselves
            """
        
        response = structured_llm.invoke([
            SystemMessage(content=system_content),
            HumanMessage(content=f"Generate Cypher for: {query}")
        ])
        
        try:
            safe_cypher = self.sanitize_cypher(response.cypher)
            if not safe_cypher:
                print(f"[Warning] Sanitization failed for query: {query}")
                return None
            
            safe_cypher = self.resolve_names_in_cypher(safe_cypher)
            
            with self.graph_driver.session() as session:
                result = session.run(safe_cypher)
                data = [record.data() for record in result]
            
            is_fallback = (
                len(data) == 1 and 
                len(data[0]) == 1 and 
                "f.path" in data[0]
            )
            
            output = {
                "is_fallback": is_fallback,
                "data": data,
                "response": response.cypher,
                "method": "llm",
                "timestamp": time.time()
            }
            
            self._graph_cache[query] = output
            return output
            
        except Exception as e:
            print(f"[Error] Graph query failed:")
            print(f"  Query: {query}")
            print(f"  Generated Cypher: {response.cypher}")
            print(f"  Exception: {type(e).__name__}: {str(e)}")
            return None
    
    def _validate_graph_result(self, result: dict | None, query: str) -> dict | None:
        """
        Validate graph results for completeness and accuracy.
        Flag incomplete or suspicious results.
        """
        if result is None:
            return None
        
        data = result.get("data", [])
        
        # Check 1: Empty result
        if not data:
            print(f"[Validation] Empty result for query: {query}")
            return result
        
        # Check 2: Single result for multi-hop query
        if len(data) == 1 and any(
            kw in query.lower() 
            for kw in ["indirectly", "transitive", "through"]
        ):
            print(f"[Validation Warning] Single result for multi-hop query: {query}")
            print(f"  Result: {data}")
            result["confidence"] = 0.3  # Low confidence
            return result
        
        # Check 3: Result variance (check if results look complete)
        if len(data) > 1:
            # For imports/dependencies, expect multiple results
            result["confidence"] = min(1.0, len(data) / 5.0)  # Assume 5+ is good
        else:
            result["confidence"] = 0.5
        
        # Check 4: Extract file count
        file_count = 0
        for record in data:
            for k, v in record.items():
                if isinstance(v, str) and v.endswith(".py"):
                    file_count += 1
                elif isinstance(v, list):
                    file_count += len([x for x in v if isinstance(x, str) and x.endswith(".py")])
        
        result["extracted_file_count"] = file_count
        
        # Check 5: Flag if result seems incomplete
        if file_count < self.min_result_count:
            print(f"[Validation Warning] Low file count: {file_count}")
            result["is_incomplete"] = True
        else:
            result["is_incomplete"] = False
        
        return result

    def _extract_filenames_safe(self, graph_result: dict | None, query: str) -> list[str]:
        """
        Safely extract filenames with validation
        """
        if not graph_result:
            return self.extract_entities_from_query(query)
        
        # Validate first
        graph_result = self._validate_graph_result(graph_result, query)
        
        if not self._is_meaningful(graph_result):
            print(f"[Fallback] Graph result not meaningful, using entity extraction")
            return self.extract_entities_from_query(query)
        
        filenames = self._extract_filenames(graph_result["data"])
        
        # If validation flagged as incomplete, augment with entity extraction
        if graph_result.get("is_incomplete"):
            print(f"[Augmentation] Graph result incomplete, augmenting with entity extraction")
            entity_files = self.extract_entities_from_query(query)
            filenames = list(set(filenames) | set(entity_files))  # Union
        
        if not filenames:
            print(f"[Fallback] No filenames extracted, using entity extraction")
            return self.extract_entities_from_query(query)
        
        print(f"[Validation] Extracted {len(filenames)} files from graph: {filenames[:3]}...")
        return filenames
    
    @property
    def embed_model(self):
        if self._embed_model is None:
            print("Loading SentenceTransformer model...")
            self._embed_model = SentenceTransformer("jinaai/jina-embeddings-v2-base-code")
        return self._embed_model

    def _load_file_index(self) -> list[dict]:
        if self._file_index is not None:
            return self._file_index

        with self.graph_driver.session() as session:
            result = session.run(f"""
                MATCH (r:Repo {{repo_id: '{self.repo_id}'}})-[:CONTAINS]->(f:File)
                OPTIONAL MATCH (f)-[:DEFINES_CLASS]->(c:Class)
                OPTIONAL MATCH (f)-[:DEFINES_FUNCTION]->(fn:Function)
                OPTIONAL MATCH (c)-[:HAS_METHOD]->(m:Function)
                RETURN f.name as name, f.path as path,
                       collect(DISTINCT c.name) as classes,
                       collect(DISTINCT fn.name) + collect(DISTINCT m.name) as functions
            """)
            self._file_index = [dict(r) for r in result]

        return self._file_index

    def resolve_filename(self, raw_name: str) -> str | None:
        index = self._load_file_index()
        names = [f["name"] for f in index]

        if raw_name in names:
            return raw_name

        match = process.extractOne(raw_name, names, scorer=fuzz.token_sort_ratio)
        if match and match[1] >= 70:
            return match[0]

        return None

    def extract_entities_from_query(self, query: str) -> list[str]:
        index = self._load_file_index()
        entity_map: dict[str, str] = {}
        
        for f in index:
            entity_map[f["name"]] = f["path"]
            for cls in (f.get("classes") or []):
                entity_map[cls] = f["path"]
            for fn in (f.get("functions") or []):
                entity_map[fn] = f["path"]

        if not entity_map:
            return []

        words = query.replace("'s", "").split()
        bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words) - 1)]
        candidates = words + bigrams

        matched_paths = []
        for candidate in candidates:
            match = process.extractOne(
                candidate,
                list(entity_map.keys()),
                scorer=fuzz.token_sort_ratio
            )
            if match and match[1] >= 75:
                matched_paths.append(entity_map[match[0]])

        return list(set(matched_paths))

    def sanitize_cypher(self, cypher: str) -> str:
        # Remove string literals to avoid false matches inside quotes
        cypher_clean = re.sub(r"'[^']*'|\"[^\"]*\"", "''", cypher)
        
        # Match property accesses: word.property (not ::, ->, etc.)
        used_props = set(re.findall(r'\b\w+\.(\w+)\b', cypher_clean))
        
        # These are valid Cypher keywords that look like properties — exclude them
        cypher_keywords = {'path', 'type', 'keys', 'labels', 'id', 'properties'}
        
        invalid = used_props - VALID_FILE_PROPERTIES - cypher_keywords
        if invalid:
            print(f"[Sanitize] Blocked invalid properties: {invalid}")
            return None
        return cypher

    def resolve_names_in_cypher(self, cypher: str) -> str:
        def replacer(match):
            raw = match.group(1)
            resolved = self.resolve_filename(raw)
            if resolved and resolved != raw:
                print(f"[resolve] '{raw}' → '{resolved}'")
                return match.group(0).replace(raw, resolved)
            return match.group(0)

        pattern = r"\{(?:name|source|via):\s*['\"]([^'\"]+)['\"]\}"
        return re.sub(pattern, replacer, cypher)

    def _is_meaningful(self, graph_result: dict) -> bool:
        if graph_result is None:
            return False
        return not graph_result.get("is_fallback", False) and bool(graph_result.get("data"))

    def vector_search(self, query: str, filenames: list, top_k: int = 5):
        qdrant_filter = Filter(
            must=[
                FieldCondition(
                    key="path",
                    match=MatchAny(any=filenames)
                )
            ]
        )
        
        return self.db_client.search(
            query=query, 
            query_filter=qdrant_filter, 
            top_k=top_k
        )

    def rerank(self, context: dict, query: str, top_k: int):
        documents = context['documents'][0]
        metadatas = context['metadatas'][0]
        
        if not documents:
            return []
        
        query_embedding = self.embed_model.encode(query, convert_to_numpy=True)
        doc_embeddings = self.embed_model.encode(documents, convert_to_numpy=True)
        
        scores = np.dot(doc_embeddings, query_embedding)
        
        ranked = sorted(
            [(metadatas[i], scores[i], documents[i]) for i in range(len(documents))],
            key=lambda x: x[1],
            reverse=True
        )[:top_k]
        
        return ranked

    def _extract_filenames(self, graph_data: list[dict]) -> list[str]:
        paths = []
        for record in graph_data:
            for v in record.values():
                if isinstance(v, str) and v.endswith(".py"):
                    paths.append(v)
                elif isinstance(v, list):
                    paths.extend(x for x in v if isinstance(x, str) and x.endswith(".py"))
        return list(set(paths))

    def _resolve_filenames(self, graph_result: dict | None, query: str) -> list[str]:
        """Updated to use validation"""
        if graph_result and self._is_meaningful(graph_result):
            filenames = self._extract_filenames(graph_result["data"])
            if filenames:
                return filenames

        print("[fallback] Graph returned no meaningful data, using entity extraction")
        return self.extract_entities_from_query(query)

    def router(self, query: str) -> RouterDecision:
        if query in self._router_cache:
            return self._router_cache[query]

        router_llm = self.llm.with_structured_output(RouterDecision)
        router_prompt = ChatPromptTemplate.from_messages([
            ('system', """You are a query router for code repository Q&A.
                The knowledge graph stores: files, imports, classes, functions, methods, inheritance, call edges.
                It has NO knowledge of variable values, runtime state, or full code behavior.

                Classify into:

                graph_only — answerable purely from code structure:
                - which files import X
                - what does <file> depend on
                - where is a class/function/method defined
                - which methods belong to a class
                - which functions/methods call or instantiate another symbol
                - which files have no imports
                - which file has the most dependencies
                - transitive dependencies of <file>
                - what files depend on <file> (reverse lookup)

                hybrid — everything else:
                - what a function/method does internally or returns
                - what happens when a condition is met
                - initial / default value of anything
                - how a feature is implemented
                - what database / framework / library is used
                - any question about runtime behavior, state, or logic

                RULE: If mentioning specific variables/fields or asking about behavior → hybrid.
                When in doubt, choose hybrid.
            """),
            ('user', "query: {query}")
        ])

        result = (router_prompt | router_llm).invoke({"query": query})
        self._router_cache[query] = result
        return result

    def get_result(self, query: str, top_k: int = 5):
        """Updated to use safe file extraction with validation"""
        route = self.router(query).decision
        retrieve_k = top_k

        def _pure_vector():
            vector_data = self.db_client.search(query=query, top_k=retrieve_k)
            return {
                "type": "hybrid",
                "graph": [],
                "vector": self.rerank(vector_data, query, top_k),
            }

        if route == "graph_only":
            graph_result = self.graph_search(query)

            if graph_result is None:
                return _pure_vector()

            if self._is_meaningful(graph_result):
                return {
                    "type": "graph",
                    "graph": graph_result["data"],
                }

            # Use SAFE extraction
            filenames = self._extract_filenames_safe(graph_result, query)

            if filenames:
                vector_data = self.vector_search(query, filenames=filenames, top_k=retrieve_k)
            else:
                vector_data = self.db_client.search(query=query, top_k=retrieve_k)

            return {
                "type": "hybrid",
                "graph": [],
                "vector": self.rerank(vector_data, query, top_k),
            }

        else:  # hybrid
            graph_result = self.graph_search(query)

            if graph_result is None:
                return _pure_vector()

            # Use SAFE extraction
            filenames = self._extract_filenames_safe(graph_result, query)
            
            if filenames:
                vector_data = self.vector_search(query, filenames=filenames, top_k=retrieve_k)
            else:
                vector_data = self.db_client.search(query=query, top_k=retrieve_k)

            return {
                "type": "hybrid",
                "graph": graph_result["data"] if graph_result else [],
                "vector": self.rerank(vector_data, query, top_k),
            }

    def close(self):
        self.graph_driver.close()
        print("Neo4j Driver close.")

p:\repo-2-graph\.venv\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [17]:
from langchain_openai import ChatOpenAI

routing_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

engine = QueryEngine(
    repo_id=filename,
    db_client=client,
    uri="bolt://localhost:7687",
    user="neo4j",
    password="pk142145",
    llm=routing_llm
)

In [18]:
def format_documents(data: dict) -> str:
    sections = []

    graph_records = data.get("graph", [])
    if graph_records:
        graph_text = ["[Graph Relationships]"]

        for i, r in enumerate(graph_records, 1):
            graph_text.append(f"\nRecord {i}:")
            for k, v in r.items():
                if v is None:
                    continue

                if isinstance(v, list):
                    values = [str(x) for x in v if x]
                    if values:
                        graph_text.append(f"  {k}: {', '.join(values)}")
                else:
                    graph_text.append(f"  {k}: {v}")

        sections.append("\n".join(graph_text))

    if data.get("type") == "hybrid":
        vector = data.get("vector", [])
        if vector:
            chunk_text = ["[Code Chunks]"]

            for meta, score, doc in vector:
                chunk_text.append(
                    f"\n[{meta.get('path', 'unknown')} | "
                    f"{meta.get('class_name', 'module_level')}.{meta.get('function_name', 'module_level_segment')} | "
                    f"lines {meta.get('line_start', '?')}-{meta.get('line_end', '?')} | "
                    f"score {score:.4f}]\n{doc}"
                )

            sections.append("\n".join(chunk_text))

    return "\n\n".join(sections).strip()

In [19]:
from pydantic import BaseModel, Field

class ResponseModel(BaseModel):
    answer: str = Field(..., description="Response of the query from the llm model")
    score: float = Field(..., description="Confidence score of the response")
    sources: str = Field(..., description="File name used for response")

class AnswerEngine:
    def __init__(self, llm):
        self.llm = llm.with_structured_output(ResponseModel)

    def generate_response(self, query: str, context: str):
        prompt = ChatPromptTemplate.from_messages([
            ("system", """
                You are a code repository assistant.
                Answer only from the provided context.
                When asked about initial or default values, prioritize code that
                constructs or initializes objects (e.g. GraphState(...), __init__,
                initial_state) over code that updates or transitions values.
                Sources should be file names used to answer.
                Confidence should be between 0 and 1.
            """),
            ("user", "Context:\n{context}\n\nQuestion: {query}")
        ])
        chain = prompt | self.llm
        return chain.invoke({"context": context, "query": query})

In [20]:
query = "Which files does app.py indirectly depend on through tutor_service.py?"
result = engine.get_result(query, 5)
formated_result = format_documents(result)

In [67]:
engine.graph_search(query)

{'is_fallback': False,
 'data': [{'indirect_dependency': 'src/database/neo4j_conn.py'},
  {'indirect_dependency': 'src/database/student_db.py'},
  {'indirect_dependency': 'src/models/diagnoses.py'},
  {'indirect_dependency': 'src/models/state.py'},
  {'indirect_dependency': 'src/services/diagnoser_service.py'},
  {'indirect_dependency': 'src/services/intent_service.py'},
  {'indirect_dependency': 'src/services/planner_service.py'},
  {'indirect_dependency': 'src/services/tutor_lesson_utils.py'}],
 'response': "\n                MATCH (r:Repo {repo_id: 'princ3kr-Notebook-LM-Mini'})\n                MATCH (source:File {name: 'app.py'})-[:IMPORTS]->(via:File {name: 'tutor_service.py'})\n                MATCH (via:File)-[:IMPORTS]->(target:File)\n                RETURN DISTINCT target.path as indirect_dependency\n                ORDER BY target.path\n            ",
 'method': 'template',
 'template_name': 'multi_hop_imports',
 'timestamp': 1779990525.9846303}

In [21]:
from langchain_openai import ChatOpenAI

answer_llm = ChatOpenAI(model="gpt-4o", temperature=0)
chat_engine = AnswerEngine(answer_llm)
result = chat_engine.generate_response(query, formated_result)

In [22]:
result.answer

'The file `app.py` indirectly depends on the following files through `tutor_service.py`:\n\n- `src/database/neo4j_conn.py`\n- `src/database/student_db.py`\n- `src/models/diagnoses.py`\n- `src/models/state.py`\n- `src/services/diagnoser_service.py`\n- `src/services/intent_service.py`\n- `src/services/planner_service.py`\n- `src/services/tutor_lesson_utils.py`'

## RAG Evaluation metrices

In [23]:
eval_questions = [
    # --- EDGE: FILES WITH NO IMPORTS ---
    {
        "question": "Which files in the repository do not import any other local module?",
        "ground_truth": "The files src/models/concept.py, src/models/state.py, src/models/diagnoses.py, and src/models/intent.py are leaf nodes in the dependency graph and do not import any local modules from the codebase."
    },

    # --- EDGE: CIRCULAR / DEEP DEPENDENCY ---
    {
        "question": "Which service has the most dependencies on other files?",
        "ground_truth": "src/services/tutor_service.py has the most dependencies as it imports Neo4jConn, StudentDB, DiagnoserService, IntentService, PlannerService, and tutor_lesson_utils."
    },

    # --- EDGE: SPECIFIC FUNCTION BEHAVIOR ---
    {
        "question": "What happens if planned_paths is empty in the planner node?",
        "ground_truth": "If planned_paths is empty, the planner node returns Command with goto=END and a final_response message explaining either that a topic was not recognized (if target_topics is empty) or that no learning path was found."
    },

    # --- EDGE: SPECIFIC CLASS METHOD ---
    {
        "question": "What does update_last_json_path do in StudentDB and when is it called?",
        "ground_truth": "update_last_json_path updates the last_json_path field and last_updated timestamp in TinyDB for the student. It is called in app.py after successfully building the knowledge graph from a PDF, and also with None if the user requests processing a new PDF."
    },

    # --- EDGE: STATE FIELD USAGE ---
    {
        "question": "What is the initial value of phase in GraphState when a new session starts?",
        "ground_truth": "The initial value of phase is 'quiz' as set in the initial_state construction in app.py."
    },

    # --- EDGE: ERROR PATH ---
    {
        "question": "What does app.py do if no concepts are extracted from the uploaded PDF?",
        "ground_truth": "app.py updates the Streamlit status to the error state, displays an error message stating that no relevant concepts were identified and to try a more detailed document, then halts execution using st.stop()."
    },

    # --- EDGE: MULTI-HOP DEPENDENCY ---
    {
        "question": "Which files does app.py indirectly depend on through tutor_service.py?",
        "ground_truth": "Through tutor_service.py, app.py indirectly depends on neo4j_conn.py, student_db.py, diagnoser_service.py, intent_service.py, planner_service.py, and tutor_lesson_utils.py."
    },

    # --- EDGE: SPECIFIC PARAMETER ---
    {
        "question": "What is the process_limit in app.py and what does it control?",
        "ground_truth": "process_limit is set to 50 in app.py and controls the maximum number of PDF chunks processed for concept extraction to avoid excessive API calls."
    },

    # --- EDGE: TWO LLM INSTANCES ---
    {
        "question": "Why does TutorWorkflow initialize two separate LLM instances?",
        "ground_truth": "TutorWorkflow uses self.llm with max_tokens=250 for quick responses like quiz questions/explanations and self.teach_llm with max_tokens=1000 and temperature=0.35 for longer concept explanations requiring more detail."
    },

    # --- EDGE: CHECKPOINT ---
    {
        "question": "What checkpointing mechanism does TutorWorkflow use and why?",
        "ground_truth": "TutorWorkflow uses MemorySaver from langgraph.checkpoint.memory as the checkpointer when compiling the workflow, which enables conversation state persistence across multiple turns using thread_id."
    },

    # --- EDGE: THREAD ID ---
    {
        "question": "How is thread_id constructed in app.py and what is it used for?",
        "ground_truth": "thread_id is constructed as f'thread_{student_id}' in app.py and is used as the LangGraph config key to isolate each student's conversation state in MemorySaver."
    },

    # --- EDGE: RESUME VS FRESH ---
    {
        "question": "What condition determines whether the tutor resumes an existing session or starts fresh?",
        "ground_truth": "If state.values and state.next both exist, the tutor resumes via Command(resume=user_input). If either is falsy, it starts fresh by invoking with a new GraphState."
    },

    # --- EDGE: TEMP FILE CLEANUP ---
    {
        "question": "How does app.py handle the temporary PDF file after processing?",
        "ground_truth": "After processing, app.py checks if the temp file exists using os.path.exists and removes it using os.remove to clean up disk space."
    },

    # --- EDGE: MISSING INPUT GUARD ---
    {
        "question": "How does app.py handle the case where user types the placeholder text 'I want to learn about...' literally?",
        "ground_truth": "app.py checks if user_input.strip().lower() equals 'i want to learn about...' or 'i want to learn about' and replaces user_input with an empty string to avoid sending the placeholder as actual input."
    },

    # --- EDGE: DATABASE LAYER ---
    {
        "question": "What database does StudentDB use internally and what file does it store data in?",
        "ground_truth": "StudentDB uses TinyDB internally and stores data in a JSON file at the path db_folder/student_sessions.json, where db_folder defaults to 'sessions'."
    },

    # --- EDGE: PLANNER SERVICE COST weights ---
    {
        "question": "What parameters determine the weights in the PlannerService cost function, and what are their default values?",
        "ground_truth": "The weights are controlled by lam1, lam2, lam3, and lam4 with default values of lam1=0.4 (mastery cost), lam2=0.3 (difficulty cost), lam3=0.2 (in-degree cost), and lam4=0.1 (out-degree benefit)."
    },

    # --- EDGE: GRAPH SERVICE FUZZY RELATIONS ---
    {
        "question": "How does GraphService build prerequisite relationships, and what similarity score threshold is required for a valid match?",
        "ground_truth": "It extracts candidate prerequisites and matches them against existing topics using process.extractOne with fuzz.token_sort_ratio. A prerequisite relationship is created only if the similarity score is 60 or higher and is not matching the concept topic itself."
    },

    # --- EDGE: INTENT SERVICE RERANK FALLBACK ---
    {
        "question": "In IntentService, what is the fallback behavior if candidate reranking using the Cross-Encoder returns no matches (returns None)?",
        "ground_truth": "If the Cross-Encoder fails to find a match above the threshold, the service falls back to the top candidate retrieved by the bi-encoder embedding model (candidates[0])."
    },

    # --- EDGE: INTENT SERVICE MODEL DEFINITION ---
    {
        "question": "What models are initialized in IntentService for candidate retrieval and reranking?",
        "ground_truth": "IntentService initializes a bi-encoder using SentenceTransformer('all-MiniLM-L6-v2') for candidate retrieval and a cross-encoder using CrossEncoder('cross-encoder/stsb-roberta-base') for candidate reranking."
    },

    # --- EDGE: PARSING EQUATIONS ---
    {
        "question": "How does tutor_lesson_utils.py parse equation fields stored in Neo4j?",
        "ground_truth": "It tries to parse the string using json.loads(s). If a JSONDecodeError occurs, it falls back to parsing with ast.literal_eval(s) to convert the string representation of list-of-dicts back into list structure."
    }
]

In [24]:
results = []

for sample in eval_questions:
    raw_result = engine.get_result(sample["question"], top_k=5)
    context_text = format_documents(raw_result)
    response = chat_engine.generate_response(sample["question"], context_text)
   
    results.append({
        "question": sample["question"],
        "answer": response.answer,
        "contexts": [context_text],
        "ground_truth": sample["ground_truth"]
    })


[Validation Warning] Low file count: 0
[Augmentation] Graph result incomplete, augmenting with entity extraction
[Validation] Extracted 1 files from graph: ['src/database/student_db.py']...


p:\repo-2-graph\.venv\Lib\site-packages\qdrant_client\common\client_warnings.py:7: UserWarning: `query` method has been deprecated and will be removed in 1.17. Instead, inference can be done internally within regular methods like `query_points` by wrapping data into `models.Document` or `models.Image`.
  warnings.warn(message, category, stacklevel=stacklevel)


Loading SentenceTransformer model...


p:\repo-2-graph\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertModel were not initialized from the model checkpoint at jinaai/jina-embeddings-v2-base-code and are newly initialized: ['embeddings.position_embeddings.weight', 'encoder.layer.0.intermediate.dense.bias', 'encoder.layer.0.intermediate.dense.weight', 'encoder.layer.0.output.LayerNorm.bias', 'encoder.layer.0.output.LayerNorm.weight', 'encoder.layer.0.output.dense.bias', 'encoder.layer.0.output.dense.weight', 'encoder.layer.1.intermediate.dense.bias', 'encoder.layer.1.intermediate.dense.weight', 'encoder.layer.1.output.LayerNorm.bias', 'encoder.layer.1.output.LayerNorm.weight', 'encoder.layer.1.output.dense.bias', 'encoder.layer.1.output.dense.weight', 'encoder.layer.10.intermediate.dense.bia

[Validation] Empty result for query: What does update_last_json_path do in StudentDB and when is it called?
[Fallback] Graph result not meaningful, using entity extraction
[Validation Warning] Low file count: 0
[Augmentation] Graph result incomplete, augmenting with entity extraction
[Validation] Extracted 2 files from graph: ['src/services/intent_service.py', 'src/models/state.py']...
[Validation Warning] Low file count: 0
[Augmentation] Graph result incomplete, augmenting with entity extraction
[Validation] Extracted 3 files from graph: ['src/models/concept.py', 'app.py', 'src/services/intent_service.py']...
[Validation] Extracted 1 files from graph: ['app.py']...
[Validation Warning] Low file count: 0
[Augmentation] Graph result incomplete, augmenting with entity extraction
[Validation] Extracted 1 files from graph: ['src/services/tutor_service.py']...
[Validation Warning] Low file count: 0
[Augmentation] Graph result incomplete, augmenting with entity extraction
[Validation] Extrac

In [25]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    answer_correctness,
    context_precision,
    context_recall
)
from ragas.llms import LangchainLLMWrapper
from datasets import Dataset

ragas_llm = LangchainLLMWrapper(ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
))

dataset = Dataset.from_list(results)

scores = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        answer_correctness,
        context_precision,
        context_recall
    ],
    llm=ragas_llm
)

print(scores)

C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_21228\339825698.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_21228\339825698.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\PRINCE KUMAR\AppData\Local\Temp\ipykernel_21228\339825698.py:2: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import (
C:\Users\PRINCE KUMAR\AppD

{'faithfulness': 0.8303, 'answer_relevancy': 0.8490, 'answer_correctness': 0.6942, 'context_precision': 0.9500, 'context_recall': 0.7750}


In [ ]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.messages import HumanMessage, AIMessage

class GraphResult(BaseModel):
    is_fallback: bool
    data: list[dict]
    method: Literal["llm", "template"]
    timestamp: float

class AgentState(BaseModel):
    repo_id: str
    current_agent: Literal["router", "graph", "vector", "synthesizer"]
    router_decision: Literal["graph", "vector", "hybrid"]
    reason: str
    plan: list[Literal["graph", "vector"]]
    user_query: str
    user_history: list[HumanMessage | AIMessage]
    cypher_query: str
    graph_result: GraphResult | None
    vector_result: list[dict]
    final_answer: str

In [ ]:
class VectorAgent:
    def __init__(self):
        